# CROME class counts

In [ ]:
#Aggregate class counts across files
import pandas as pd
import geopandas as gpd
import os
from pathlib import Path
data_path = '/content/drive/MyDrive/Dissertation/crome/2022/crome_2022_refined/east_anglia'
files = [f for f in os.listdir(data_path) if f.endswith(('.shp', '.csv', '.geojson'))]
all_counts = []
first_row_printed = False
target_col = 'ukceh_aligned_name'
for file in files:
    file_path = os.path.join(data_path, file)
    try:
        if file.endswith('.csv'):
            df = pd.read_csv(file_path)
        else:
            df = gpd.read_file(file_path)
        if not first_row_printed:
            print(f"--- Processing: {file} ---")
            display(df.head(1))
            first_row_printed = True
        if target_col in df.columns:
            counts = df[target_col].value_counts()
            all_counts.append(counts)
        else:
            print(f"Warning: {file} lacks column '{target_col}'.")
    except Exception as e:
        print(f"Error reading {file}: {e}")
if all_counts:
    total_stats = pd.concat(all_counts).groupby(level=0).sum().sort_values(ascending=False)
    print(f"\n--- Counts by {target_col} ---")
    display(total_stats)
else:
    print(f"Could not count column '{target_col}'.")

In [ ]:
#build a observed class mapping lookuptable
data_path = '/content/drive/MyDrive/Dissertation/crome/2022/crome_2022_refined/east_anglia'
lookup_cols = ['lucode', 'crome_original_name', 'ukceh_aligned_name', 'analysis_class']
files = [f for f in os.listdir(data_path) if f.endswith(('.shp', '.csv', '.geojson'))]
all_mappings = []
for file in files:
    file_path = os.path.join(data_path, file)
    try:
        if file.endswith('.csv'):
            df = pd.read_csv(file_path)
        else:
            df = gpd.read_file(file_path)
        available_cols = [col for col in lookup_cols if col in df.columns]
        if available_cols:
            unique_combinations = df[available_cols].drop_duplicates()
            all_mappings.append(unique_combinations)
    except Exception as e:
        print(f"Error processing {file}: {e}")
if all_mappings:
    lookup_table = pd.concat(all_mappings).drop_duplicates().sort_values(by='lucode' if 'lucode' in lookup_cols else lookup_cols[0])
    print("--- Crop lookup table ---")
    display(lookup_table)
    # lookup_table.to_csv('/content/crop_lookup_table.csv', index=False)
else:
    print("No valid lookup data.")

In [ ]:
#Export the table.
lookup_table.to_csv('/content/crop_lookup_table.csv', index=False)